# 3.7 学習と検証の実施

- 本ファイルでは、PSPNetの学習と検証の実施を行います。AWSのGPUマシンで計算します。
- p2.xlargeで約12時間かかります。


# 学習目標

1.	PSPNetの学習と検証を実装できるようになる
2.	セマンティックセグメンテーションのファインチューニングを理解する


# 事前準備

- 本書に従い学習済みモデルのファイル「pspnet50_ADE20K.pth」をダウンロードし、フォルダ「weights」に用意します。

In [1]:
# パッケージのimport
import random
import math
import time
import pandas as pd
import numpy as np

import torch
import torch.utils.data as data
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim

In [3]:
# 初期設定
# Setup seeds
torch.manual_seed(1234)
np.random.seed(1234)
random.seed(1234)

# DataLoader作成

In [4]:
from utils.dataloader import make_datapath_list, DataTransform, VOCDataset

# ファイルパスリスト作成
rootpath = "./data/VOCdevkit/VOC2012/"
train_img_list, train_anno_list, val_img_list, val_anno_list = make_datapath_list(
    rootpath=rootpath)

# Dataset作成
# (RGB)の色の平均値と標準偏差
color_mean = (0.485, 0.456, 0.406)
color_std = (0.229, 0.224, 0.225)

train_dataset = VOCDataset(train_img_list, train_anno_list, phase="train", transform=DataTransform(
    input_size=475, color_mean=color_mean, color_std=color_std))

val_dataset = VOCDataset(val_img_list, val_anno_list, phase="val", transform=DataTransform(
    input_size=475, color_mean=color_mean, color_std=color_std))

# DataLoader作成
batch_size = 4

train_dataloader = data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True)

val_dataloader = data.DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False)

# 辞書型変数にまとめる
dataloaders_dict = {"train": train_dataloader, "val": val_dataloader}


# ネットワークモデル作成

In [5]:
from utils.pspnet import PSPNet

# ファインチューニングでPSPNetを作成
# ADE20Kデータセットの学習済みモデルを使用、ADE20Kはクラス数が150です
net = PSPNet(n_classes=150)

# ADE20K学習済みパラメータをロード
state_dict = torch.load("./weights/pspnet50_ADE20K.pth")
net.load_state_dict(state_dict)

# 分類用の畳み込み層を、出力数21のものにつけかえる
n_classes = 21
net.decode_feature.classification = nn.Conv2d(
    in_channels=512, out_channels=n_classes, kernel_size=1, stride=1, padding=0)

net.aux.classification = nn.Conv2d(
    in_channels=256, out_channels=n_classes, kernel_size=1, stride=1, padding=0)

# 付け替えた畳み込み層を初期化する。活性化関数がシグモイド関数なのでXavierを使用する。


def weights_init(m):
    if isinstance(m, nn.Conv2d):
        nn.init.xavier_normal_(m.weight.data)
        if m.bias is not None:  # バイアス項がある場合
            nn.init.constant_(m.bias, 0.0)


net.decode_feature.classification.apply(weights_init)
net.aux.classification.apply(weights_init)


print('ネットワーク設定完了：学習済みの重みをロードしました')


ネットワーク設定完了：学習済みの重みをロードしました


In [6]:
net

PSPNet(
  (feature_conv): FeatureMap_convolution(
    (cbnr_1): conv2DBatchNormRelu(
      (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (batchnorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace)
    )
    (cbnr_2): conv2DBatchNormRelu(
      (conv): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (batchnorm): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace)
    )
    (cbnr_3): conv2DBatchNormRelu(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (batchnorm): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace)
    )
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (feature_res_1): ResidualBlockPSP(
    (block1): bottleNeckPSP(
      (cb

# 損失関数を定義

In [7]:
# 損失関数の設定
class PSPLoss(nn.Module):
    """PSPNetの損失関数のクラスです。"""

    def __init__(self, aux_weight=0.4):
        super(PSPLoss, self).__init__()
        self.aux_weight = aux_weight  # aux_lossの重み

    def forward(self, outputs, targets):
        """
        損失関数の計算。

        Parameters
        ----------
        outputs : PSPNetの出力(tuple)
            (output=torch.Size([num_batch, 21, 475, 475]), output_aux=torch.Size([num_batch, 21, 475, 475]))。

        targets : [num_batch, 475, 4755]
            正解のアノテーション情報

        Returns
        -------
        loss : テンソル
            損失の値
        """

        loss = F.cross_entropy(outputs[0], targets, reduction='mean')
        loss_aux = F.cross_entropy(outputs[1], targets, reduction='mean')

        return loss+self.aux_weight*loss_aux


criterion = PSPLoss(aux_weight=0.4)


# 最適化手法を設定

In [8]:
# ファインチューニングなので、学習率は小さく
optimizer = optim.SGD([
    {'params': net.feature_conv.parameters(), 'lr': 1e-3},
    {'params': net.feature_res_1.parameters(), 'lr': 1e-3},
    {'params': net.feature_res_2.parameters(), 'lr': 1e-3},
    {'params': net.feature_dilated_res_1.parameters(), 'lr': 1e-3},
    {'params': net.feature_dilated_res_2.parameters(), 'lr': 1e-3},
    {'params': net.pyramid_pooling.parameters(), 'lr': 1e-3},
    {'params': net.decode_feature.parameters(), 'lr': 1e-2},
    {'params': net.aux.parameters(), 'lr': 1e-2},
], momentum=0.9, weight_decay=0.0001)


# スケジューラーの設定
def lambda_epoch(epoch):
    max_epoch = 30
    return math.pow((1-epoch/max_epoch), 0.9)


scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda_epoch)


# 学習・検証を実施する

In [9]:
# モデルを学習させる関数を作成


def train_model(net, dataloaders_dict, criterion, scheduler, optimizer, num_epochs):

    # GPUが使えるかを確認
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("使用デバイス：", device)

    # ネットワークをGPUへ
    net.to(device)

    # ネットワークがある程度固定であれば、高速化させる
    torch.backends.cudnn.benchmark = True

    # 画像の枚数
    num_train_imgs = len(dataloaders_dict["train"].dataset)
    num_val_imgs = len(dataloaders_dict["val"].dataset)
    batch_size = dataloaders_dict["train"].batch_size

    # イテレーションカウンタをセット
    iteration = 1
    logs = []

    # multiple minibatch
    batch_multiplier = 2
    #batch_multiplier = 3
    # epochのループ
    for epoch in range(num_epochs):

        # 開始時刻を保存
        t_epoch_start = time.time()
        t_iter_start = time.time()
        epoch_train_loss = 0.0  # epochの損失和
        epoch_val_loss = 0.0  # epochの損失和

        print('-------------')
        print('Epoch {}/{}'.format(epoch+1, num_epochs))
        print('-------------')

        # epochごとの訓練と検証のループ
        for phase in ['train', 'val']:
            if phase == 'train':
                net.train()  # モデルを訓練モードに
                scheduler.step()  # 最適化schedulerの更新
                optimizer.zero_grad()
                print('（train）')

            else:
                if((epoch+1) % 5 == 0):
                    net.eval()   # モデルを検証モードに
                    print('-------------')
                    print('（val）')
                else:
                    # 検証は5回に1回だけ行う
                    continue

            # データローダーからminibatchずつ取り出すループ
            count = 0  # multiple minibatch
            for imges, anno_class_imges in dataloaders_dict[phase]:
                # ミニバッチがサイズが1だと、バッチノーマライゼーションでエラーになるのでさける
                if imges.size()[0] == 1:
                    continue

                # GPUが使えるならGPUにデータを送る
                imges = imges.to(device)
                anno_class_imges = anno_class_imges.to(device)

                
                # multiple minibatchでのパラメータの更新
                if (phase == 'train') and (count == 0):
                    optimizer.step()
                    optimizer.zero_grad()
                    count = batch_multiplier

                # 順伝搬（forward）計算
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = net(imges)
                    loss = criterion(
                        outputs, anno_class_imges.long()) / batch_multiplier

                    # 訓練時はバックプロパゲーション
                    if phase == 'train':
                        loss.backward()  # 勾配の計算
                        count -= 1  # multiple minibatch

                        if (iteration % 10 == 0):  # 10iterに1度、lossを表示
                            t_iter_finish = time.time()
                            duration = t_iter_finish - t_iter_start
                            print('イテレーション {} || Loss: {:.4f} || 10iter: {:.4f} sec.'.format(
                                iteration, loss.item()/batch_size*batch_multiplier, duration))
                            t_iter_start = time.time()

                        epoch_train_loss += loss.item() * batch_multiplier
                        iteration += 1

                    # 検証時
                    else:
                        epoch_val_loss += loss.item() * batch_multiplier

        # epochのphaseごとのlossと正解率
        t_epoch_finish = time.time()
        print('-------------')
        print('epoch {} || Epoch_TRAIN_Loss:{:.4f} ||Epoch_VAL_Loss:{:.4f}'.format(
            epoch+1, epoch_train_loss/num_train_imgs, epoch_val_loss/num_val_imgs))
        print('timer:  {:.4f} sec.'.format(t_epoch_finish - t_epoch_start))
        t_epoch_start = time.time()

        # ログを保存
        #log_epoch = {'epoch': epoch+1, 'train_loss': epoch_train_loss /
        #             num_train_imgs, 'val_loss': epoch_val_loss/num_val_imgs}
        #logs.append(log_epoch)
        #df = pd.DataFrame(logs)
        #df.to_csv("log_output.csv")

    # 最後のネットワークを保存する
    torch.save(net.state_dict(), 'weights/pspnet50_' +
               str(epoch+1) + '.pth')


In [10]:
# 学習・検証を実行する
num_epochs = 30
train_model(net, dataloaders_dict, criterion, scheduler, optimizer, num_epochs=num_epochs)


使用デバイス： cuda:0
-------------
Epoch 1/30
-------------
（train）
イテレーション 10 || Loss: 0.4846 || 10iter: 9.2601 sec.
イテレーション 20 || Loss: 1.0447 || 10iter: 4.3557 sec.
イテレーション 30 || Loss: 0.6342 || 10iter: 4.5854 sec.
イテレーション 40 || Loss: 0.2750 || 10iter: 4.5086 sec.
イテレーション 50 || Loss: 0.3135 || 10iter: 4.4261 sec.
イテレーション 60 || Loss: 0.2291 || 10iter: 4.4754 sec.
イテレーション 70 || Loss: 0.1917 || 10iter: 4.5527 sec.
イテレーション 80 || Loss: 0.2384 || 10iter: 4.6299 sec.
イテレーション 90 || Loss: 0.1601 || 10iter: 4.4103 sec.
イテレーション 100 || Loss: 0.1418 || 10iter: 4.3926 sec.
イテレーション 110 || Loss: 0.2928 || 10iter: 4.4309 sec.
イテレーション 120 || Loss: 0.0770 || 10iter: 4.4226 sec.
イテレーション 130 || Loss: 0.2365 || 10iter: 4.4524 sec.
イテレーション 140 || Loss: 0.2093 || 10iter: 4.4188 sec.
イテレーション 150 || Loss: 0.3377 || 10iter: 4.4403 sec.
イテレーション 160 || Loss: 0.2474 || 10iter: 4.5487 sec.
イテレーション 170 || Loss: 0.4981 || 10iter: 4.5482 sec.
イテレーション 180 || Loss: 0.2147 || 10iter: 4.4505 sec.
イテレーション 190 || Loss: 0.2132 |

イテレーション 1490 || Loss: 0.2143 || 10iter: 4.4904 sec.
イテレーション 1500 || Loss: 0.2550 || 10iter: 4.5246 sec.
イテレーション 1510 || Loss: 0.0967 || 10iter: 4.5281 sec.
イテレーション 1520 || Loss: 0.2171 || 10iter: 4.4767 sec.
イテレーション 1530 || Loss: 0.1010 || 10iter: 4.5036 sec.
イテレーション 1540 || Loss: 0.0585 || 10iter: 4.5453 sec.
イテレーション 1550 || Loss: 0.1120 || 10iter: 4.4894 sec.
イテレーション 1560 || Loss: 0.0705 || 10iter: 4.5138 sec.
イテレーション 1570 || Loss: 0.1213 || 10iter: 4.4869 sec.
イテレーション 1580 || Loss: 0.0958 || 10iter: 4.5061 sec.
イテレーション 1590 || Loss: 0.1247 || 10iter: 4.5146 sec.
イテレーション 1600 || Loss: 0.0461 || 10iter: 4.5164 sec.
イテレーション 1610 || Loss: 0.1041 || 10iter: 4.4950 sec.
イテレーション 1620 || Loss: 0.0479 || 10iter: 4.4909 sec.
イテレーション 1630 || Loss: 0.0888 || 10iter: 4.5215 sec.
イテレーション 1640 || Loss: 0.1354 || 10iter: 4.5173 sec.
イテレーション 1650 || Loss: 0.0398 || 10iter: 4.5012 sec.
イテレーション 1660 || Loss: 0.0862 || 10iter: 4.5057 sec.
イテレーション 1670 || Loss: 0.1242 || 10iter: 4.5073 sec.
イテレーション 1680

イテレーション 2960 || Loss: 0.0289 || 10iter: 4.4930 sec.
イテレーション 2970 || Loss: 0.0677 || 10iter: 4.4933 sec.
イテレーション 2980 || Loss: 0.0851 || 10iter: 4.5062 sec.
イテレーション 2990 || Loss: 0.1249 || 10iter: 4.4811 sec.
イテレーション 3000 || Loss: 0.0700 || 10iter: 4.5007 sec.
イテレーション 3010 || Loss: 0.0731 || 10iter: 4.4805 sec.
イテレーション 3020 || Loss: 0.0968 || 10iter: 4.5184 sec.
イテレーション 3030 || Loss: 0.1025 || 10iter: 4.4770 sec.
イテレーション 3040 || Loss: 0.0551 || 10iter: 4.4914 sec.
イテレーション 3050 || Loss: 0.0785 || 10iter: 4.4769 sec.
イテレーション 3060 || Loss: 0.2456 || 10iter: 4.5165 sec.
イテレーション 3070 || Loss: 0.0632 || 10iter: 4.4919 sec.
イテレーション 3080 || Loss: 0.1063 || 10iter: 4.4769 sec.
イテレーション 3090 || Loss: 0.3095 || 10iter: 4.5032 sec.
イテレーション 3100 || Loss: 0.2275 || 10iter: 4.5072 sec.
イテレーション 3110 || Loss: 0.1466 || 10iter: 4.4994 sec.
イテレーション 3120 || Loss: 0.1360 || 10iter: 4.5093 sec.
イテレーション 3130 || Loss: 0.1546 || 10iter: 4.4993 sec.
イテレーション 3140 || Loss: 0.1029 || 10iter: 4.4728 sec.
イテレーション 3150

イテレーション 4430 || Loss: 0.0589 || 10iter: 4.5255 sec.
イテレーション 4440 || Loss: 0.1262 || 10iter: 4.4713 sec.
イテレーション 4450 || Loss: 0.1098 || 10iter: 4.5171 sec.
イテレーション 4460 || Loss: 0.1077 || 10iter: 4.5007 sec.
イテレーション 4470 || Loss: 0.0442 || 10iter: 4.4905 sec.
イテレーション 4480 || Loss: 0.0732 || 10iter: 4.5388 sec.
イテレーション 4490 || Loss: 0.1159 || 10iter: 4.5089 sec.
イテレーション 4500 || Loss: 0.0599 || 10iter: 4.4777 sec.
イテレーション 4510 || Loss: 0.0237 || 10iter: 4.4767 sec.
イテレーション 4520 || Loss: 0.0847 || 10iter: 4.5003 sec.
イテレーション 4530 || Loss: 0.0405 || 10iter: 4.4898 sec.
イテレーション 4540 || Loss: 0.0644 || 10iter: 4.5221 sec.
イテレーション 4550 || Loss: 0.0714 || 10iter: 4.4779 sec.
イテレーション 4560 || Loss: 0.0675 || 10iter: 4.6563 sec.
イテレーション 4570 || Loss: 0.0421 || 10iter: 4.6964 sec.
イテレーション 4580 || Loss: 0.0368 || 10iter: 4.5028 sec.
イテレーション 4590 || Loss: 0.1204 || 10iter: 4.5115 sec.
イテレーション 4600 || Loss: 0.0887 || 10iter: 4.4845 sec.
イテレーション 4610 || Loss: 0.0539 || 10iter: 4.4785 sec.
イテレーション 4620

イテレーション 5900 || Loss: 0.1217 || 10iter: 4.5449 sec.
イテレーション 5910 || Loss: 0.0460 || 10iter: 4.5182 sec.
イテレーション 5920 || Loss: 0.1242 || 10iter: 4.5136 sec.
イテレーション 5930 || Loss: 0.0277 || 10iter: 4.5910 sec.
イテレーション 5940 || Loss: 0.0687 || 10iter: 4.4740 sec.
イテレーション 5950 || Loss: 0.0623 || 10iter: 4.4900 sec.
イテレーション 5960 || Loss: 0.1020 || 10iter: 4.5225 sec.
イテレーション 5970 || Loss: 0.0588 || 10iter: 4.5011 sec.
イテレーション 5980 || Loss: 0.2722 || 10iter: 4.5229 sec.
イテレーション 5990 || Loss: 0.1100 || 10iter: 4.5406 sec.
イテレーション 6000 || Loss: 0.0796 || 10iter: 4.4888 sec.
イテレーション 6010 || Loss: 0.0452 || 10iter: 4.4963 sec.
イテレーション 6020 || Loss: 0.0530 || 10iter: 4.5096 sec.
イテレーション 6030 || Loss: 0.0588 || 10iter: 4.5209 sec.
イテレーション 6040 || Loss: 0.0467 || 10iter: 4.4983 sec.
イテレーション 6050 || Loss: 0.0553 || 10iter: 4.5786 sec.
イテレーション 6060 || Loss: 0.0568 || 10iter: 4.4990 sec.
イテレーション 6070 || Loss: 0.0468 || 10iter: 4.5652 sec.
イテレーション 6080 || Loss: 0.0628 || 10iter: 4.4854 sec.
イテレーション 6090

イテレーション 7370 || Loss: 0.0526 || 10iter: 4.4889 sec.
イテレーション 7380 || Loss: 0.0347 || 10iter: 4.5069 sec.
イテレーション 7390 || Loss: 0.1059 || 10iter: 4.4949 sec.
イテレーション 7400 || Loss: 0.0281 || 10iter: 4.4941 sec.
イテレーション 7410 || Loss: 0.0487 || 10iter: 4.5196 sec.
イテレーション 7420 || Loss: 0.1115 || 10iter: 4.4922 sec.
イテレーション 7430 || Loss: 0.1278 || 10iter: 4.5051 sec.
イテレーション 7440 || Loss: 0.0610 || 10iter: 4.4776 sec.
イテレーション 7450 || Loss: 0.0412 || 10iter: 4.4806 sec.
イテレーション 7460 || Loss: 0.1430 || 10iter: 4.5043 sec.
イテレーション 7470 || Loss: 0.0464 || 10iter: 4.4794 sec.
イテレーション 7480 || Loss: 0.0617 || 10iter: 4.5078 sec.
イテレーション 7490 || Loss: 0.0510 || 10iter: 4.5068 sec.
イテレーション 7500 || Loss: 0.0594 || 10iter: 4.5206 sec.
イテレーション 7510 || Loss: 0.1372 || 10iter: 4.4729 sec.
イテレーション 7520 || Loss: 0.0578 || 10iter: 4.4803 sec.
イテレーション 7530 || Loss: 0.0172 || 10iter: 4.4655 sec.
イテレーション 7540 || Loss: 0.1017 || 10iter: 4.5079 sec.
イテレーション 7550 || Loss: 0.0767 || 10iter: 4.5067 sec.
イテレーション 7560

イテレーション 8840 || Loss: 0.0328 || 10iter: 4.4976 sec.
イテレーション 8850 || Loss: 0.0964 || 10iter: 4.5115 sec.
イテレーション 8860 || Loss: 0.0754 || 10iter: 4.5062 sec.
イテレーション 8870 || Loss: 0.1298 || 10iter: 4.5031 sec.
イテレーション 8880 || Loss: 0.0557 || 10iter: 4.4958 sec.
イテレーション 8890 || Loss: 0.0200 || 10iter: 4.5112 sec.
イテレーション 8900 || Loss: 0.0347 || 10iter: 4.4890 sec.
イテレーション 8910 || Loss: 0.1265 || 10iter: 4.5498 sec.
イテレーション 8920 || Loss: 0.0975 || 10iter: 4.5035 sec.
イテレーション 8930 || Loss: 0.1355 || 10iter: 4.5041 sec.
イテレーション 8940 || Loss: 0.0371 || 10iter: 4.4927 sec.
イテレーション 8950 || Loss: 0.0484 || 10iter: 4.5019 sec.
イテレーション 8960 || Loss: 0.0451 || 10iter: 4.5166 sec.
イテレーション 8970 || Loss: 0.0672 || 10iter: 4.4823 sec.
イテレーション 8980 || Loss: 0.0373 || 10iter: 4.4863 sec.
イテレーション 8990 || Loss: 0.1230 || 10iter: 4.5138 sec.
イテレーション 9000 || Loss: 0.0695 || 10iter: 4.4870 sec.
イテレーション 9010 || Loss: 0.0719 || 10iter: 4.4857 sec.
イテレーション 9020 || Loss: 0.0518 || 10iter: 4.5759 sec.
イテレーション 9030

イテレーション 10300 || Loss: 0.0509 || 10iter: 4.5019 sec.
イテレーション 10310 || Loss: 0.0418 || 10iter: 4.5059 sec.
イテレーション 10320 || Loss: 0.0854 || 10iter: 4.5096 sec.
イテレーション 10330 || Loss: 0.0784 || 10iter: 4.5100 sec.
イテレーション 10340 || Loss: 0.1009 || 10iter: 4.5253 sec.
イテレーション 10350 || Loss: 0.0744 || 10iter: 4.4955 sec.
イテレーション 10360 || Loss: 0.0774 || 10iter: 4.5235 sec.
イテレーション 10370 || Loss: 0.0878 || 10iter: 4.5033 sec.
イテレーション 10380 || Loss: 0.1252 || 10iter: 4.5130 sec.
イテレーション 10390 || Loss: 0.0385 || 10iter: 4.4917 sec.
イテレーション 10400 || Loss: 0.0847 || 10iter: 4.5041 sec.
イテレーション 10410 || Loss: 0.1018 || 10iter: 4.5086 sec.
イテレーション 10420 || Loss: 0.0467 || 10iter: 4.6396 sec.
イテレーション 10430 || Loss: 0.0644 || 10iter: 4.4759 sec.
イテレーション 10440 || Loss: 0.0607 || 10iter: 4.5058 sec.
イテレーション 10450 || Loss: 0.1481 || 10iter: 4.5110 sec.
イテレーション 10460 || Loss: 0.0603 || 10iter: 4.4848 sec.
イテレーション 10470 || Loss: 0.0686 || 10iter: 4.5295 sec.
イテレーション 10480 || Loss: 0.1178 || 10iter: 4.494

In [15]:
%reset out
%reset array

Once deleted, variables cannot be recovered. Proceed (y/[n])? y
Flushing output cache (0 entries)
Once deleted, variables cannot be recovered. Proceed (y/[n])? y


In [10]:
import sys

print("{}{: >25}{}{: >10}{}".format('|','Variable Name','|','Memory','|'))
print(" ------------------------------------ ")
for var_name in dir():
    if not var_name.startswith("_"):
        print("{}{: >25}{}{: >10}{}".format('|',var_name,'|',sys.getsizeof(eval(var_name)),'|'))


|            Variable Name|    Memory|
 ------------------------------------ 
|            DataTransform|      1056|
|                        F|        80|
|                       In|       192|
|                      Out|       240|
|                  PSPLoss|      1464|
|                   PSPNet|      1184|
|               VOCDataset|      1056|
|               batch_size|        28|
|               color_mean|        72|
|                color_std|        72|
|                criterion|        56|
|                     data|        80|
|         dataloaders_dict|       240|
|                     exit|        56|
|              get_ipython|        64|
|                     init|        80|
|             lambda_epoch|       136|
|       make_datapath_list|       136|
|                     math|        80|
|                n_classes|        28|
|                      net|        56|
|                       nn|        80|
|                       np|        80|
|               num_epoch

以上